> **Release dependency:** the original `clinical_review/` helper package was absent from the supplied source tree. This notebook is retained for documentation but cannot run until those original helpers are restored. See `docs/REPRODUCIBILITY.md`. No completed reviewer data are distributed.

# BR08S — Create blank offline Shaoxing report ratings
Run AFTER BR08 releases reports, BEFORE offline rating. CPU only. Creates no clinical judgments. Existing forms are preserved. Download PUBLIC; never share PRIVATE. Use BR08AS to import this package, because the original BR08A pins the internal-validation images.


In [ ]:
from pathlib import Path
import sys
EXP = Path('/path/to/repository/experiment02')
REVIEW_DIR = Path('/path/to/repository/outputs/real_run_001/experiment02_v1/context_fit/v1/shaoxing/clinical_review/shaoxing/calibration_v1')
sys.path.insert(0, str(EXP))
from common import read, sha, signature, immutable, lock
from clinical_review.workflow import _check, RATING_FIELDS
protocol = _check(REVIEW_DIR)
PUBLIC = REVIEW_DIR / 'PUBLIC'
if not (REVIEW_DIR / 'PRIVATE/references_submitted.json').is_file():
    raise ValueError('Run BR08 reference submission/report release first')
reports = read(PUBLIC / 'reports.json')
assert reports and len({r['review_id'] for r in reports}) == len(reports)
print('Released reports:', len(reports))


In [ ]:
# Pin the source BEFORE human editing. Importer checks these protected values later.
identity = dict(reports_signature=signature(reports),
    image_hashes={r['image']: sha(PUBLIC / r['image']) for r in reports},
    package_protocol_sha256=sha(REVIEW_DIR / 'PRIVATE/protocol.json'))
blank = dict(reviewer_id=protocol['reviewer_id'], agreement=None,
    clinically_significant_error_count=None, clinically_significant_omission_count=None,
    minor_error_count=None, overall_quality_1_to_5=None, uninterpretable_image=None,
    discrepancy_details='', comments='')
if set(blank) != set(RATING_FIELDS):
    raise ValueError('Native rating schema differs; inspect before creating the form')
form = dict(schema_version='blinded_report_ratings_v1', stage=protocol['stage'],
    source_reports_file='reports.json', expected_reports=len(reports),
    ratings=[dict(review_id=r['review_id'], **blank) for r in reports])
import json
with lock(REVIEW_DIR / 'offline_rating_form.lock'):
    immutable(REVIEW_DIR / 'PRIVATE/offline_rating_source.json', identity)
    target = PUBLIC / 'report_ratings_completed.json'
    if target.exists():
        existing = read(target)
        if existing.get('schema_version') != form['schema_version'] or [r['review_id'] for r in existing.get('ratings', [])] != [r['review_id'] for r in form['ratings']]:
            raise ValueError('Existing form belongs to a different package; do not overwrite')
        print('Existing form preserved; no ratings changed:', target)
    else:
        with target.open('x', encoding='utf-8') as f:
            json.dump(form, f, indent=2)
            f.write('\n')
        print('Blank form created:', target)
print('Download PUBLIC: images, reports.json, reference_interpretations.json, instructions and rating form.')
print('Complete every rating manually. Upload to the same path, then run BR08AS.')


## Human entry
Match each `review_id` to `reports.json`, then use its image and your frozen reference interpretation. Keep IDs and metadata unchanged. Set agreement to `agree`, `disagree` or `uncertain`; assign nonnegative integer counts; quality 1–5; and boolean uninterpretable_image. Explain discrepancies and uncertainty. Null means unassessed, not zero. Consult SHAOXING_HUMAN_REVIEW_INSTRUCTIONS.txt.

After upload run **BR08AS_import_shaoxing_report_ratings.ipynb**. It freezes initial ratings and creates the adjudication queue. Do not use the old internal-validation BR08A unchanged.
